# Kanka Character Updater
Notebook generato automaticamente dalla codebase Apps Script.
Ogni sezione contiene il contenuto originale dei file .js.


## Main.js


In [ ]:
const SUPPORTED_ENTITY_TYPES = [
  'characters',
  'locations',
  'families',
  'organisations',
  'races',
  'creatures',
  'items',
  'events',
  'journals',
  'quests',
  'abilities',
  'calendars',
  'timelines'
];

const REFERENCE_ENTITY_TYPES = [
  'locations',
  'families',
  'organisations',
  'races',
  'creatures',
  'items',
  'events',
  'journals',
  'quests',
  'abilities',
  'calendars',
  'timelines',
  'tags'
];

const SYNC_STATE_KEY = 'KANKA_SYNC_STATE';
const SUMMARY_PLACEHOLDER = '[[KANKA_SYNC_SUMMARY]]';
const MIN_REMAINING_SECONDS = 60;

function syncAllEntitiesToOneDocument() {
  runFullSync();
}

function continueSyncAllEntities() {
  runFullSync();
}

function runFullSync() {
  checkRequiredConfig();

  const docId = getMasterDocumentId();
  if (!docId) {
    Logger.log('[ERRORE] Documento principale non configurato.');
    return;
  }

  const doc = DocumentApp.openById(docId);
  const body = doc.getBody();

  let state = loadSyncState(docId);
  if (!state.docInitialized) {
    initializeSyncDocument(body);
    state.docInitialized = true;
    saveSyncState(state);
  }

  preloadReferenceCaches();

  const completed = processEntityTypes(doc, state);
  saveSyncState(state);

  if (completed) {
    finalizeSync(doc, state);
    clearSyncState();
    cleanupContinuationTriggers();
    Logger.log(`[OK] Documento aggiornato: https://docs.google.com/document/d/${docId}/edit`);
    return;
  }

  scheduleContinuation();
}

function processEntityTypes(doc, state) {
  const body = doc.getBody();

  while (state.typeIndex < SUPPORTED_ENTITY_TYPES.length) {
    if (shouldYield()) {
      return false;
    }

    const currentType = SUPPORTED_ENTITY_TYPES[state.typeIndex];
    const entities = fetchAllEntities(currentType);
    rememberEntityNames(currentType, entities);

    if (!(currentType in state.counts)) {
      state.counts[currentType] = entities.length;
      saveSyncState(state);
    }

    if (!state.typeHeaderInserted) {
      appendTypeHeading(body, currentType);
      state.typeHeaderInserted = true;
      saveSyncState(state);
    }

    for (let index = state.entityIndex; index < entities.length; index++) {
      renderEntityByType(currentType, entities[index], doc);
      state.entityIndex = index + 1;

      if (shouldYield()) {
        return false;
      }
    }

    body.appendParagraph('');
    state.typeIndex += 1;
    state.entityIndex = 0;
    state.typeHeaderInserted = false;
    saveSyncState(state);
  }

  return true;
}

function appendTypeHeading(body, type) {
  body.appendParagraph(capitalize(type))
      .setHeading(DocumentApp.ParagraphHeading.HEADING1);
  body.appendParagraph('');
}

function preloadReferenceCaches() {
  preloadEntityMaps(REFERENCE_ENTITY_TYPES);
}

function shouldYield() {
  try {
    return Utilities.getRemainingExecutionTime() < MIN_REMAINING_SECONDS;
  } catch (error) {
    Logger.log(`[WARN] Impossibile verificare il tempo rimanente: ${error}`);
    return false;
  }
}

function scheduleContinuation() {
  Logger.log('[INFO] Tempo quasi esaurito. Pianifico una nuova esecuzione per completare la sincronizzazione.');
  cleanupContinuationTriggers();
  ScriptApp.newTrigger('continueSyncAllEntities')
    .timeBased()
    .after(1 * 60 * 1000)
    .create();
}

function cleanupContinuationTriggers() {
  const triggers = ScriptApp.getProjectTriggers();
  triggers.forEach(trigger => {
    if (trigger.getHandlerFunction && trigger.getHandlerFunction() === 'continueSyncAllEntities') {
      ScriptApp.deleteTrigger(trigger);
    }
  });
}

function loadSyncState(docId) {
  const props = PropertiesService.getScriptProperties();
  const raw = props.getProperty(SYNC_STATE_KEY);

  if (!raw) {
    return createInitialState(docId);
  }

  try {
    const state = JSON.parse(raw);
    if (state.docId !== docId) {
      return createInitialState(docId);
    }
    state.counts = state.counts || {};
    return state;
  } catch (error) {
    Logger.log(`[WARN] Stato di sincronizzazione non valido. Riparto da zero: ${error}`);
    return createInitialState(docId);
  }
}

function saveSyncState(state) {
  const props = PropertiesService.getScriptProperties();
  props.setProperty(SYNC_STATE_KEY, JSON.stringify(state));
}

function clearSyncState() {
  PropertiesService.getScriptProperties().deleteProperty(SYNC_STATE_KEY);
}

function createInitialState(docId) {
  return {
    docId,
    docInitialized: false,
    typeIndex: 0,
    entityIndex: 0,
    typeHeaderInserted: false,
    counts: {}
  };
}

function initializeSyncDocument(body) {
  body.clear();
  const firstChild = body.getChild(0);
  if (firstChild && firstChild.getType() === DocumentApp.ElementType.PARAGRAPH) {
    firstChild.asParagraph().clear();
  }

  body.appendParagraph('Enciclopedia Kanka')
      .setHeading(DocumentApp.ParagraphHeading.HEADING1);
  body.appendParagraph(`Aggiornato il: ${new Date().toLocaleString()}`);
  body.appendParagraph('');
  body.appendParagraph(SUMMARY_PLACEHOLDER);
  body.appendParagraph('');
}

function finalizeSync(doc, state) {
  injectSummary(doc, state.counts || {});
}

function injectSummary(doc, counts) {
  const body = doc.getBody();
  const summaryRows = SUPPORTED_ENTITY_TYPES.map(type => [
    capitalize(type),
    String(counts[type] || 0)
  ]);

  let placeholderRange = body.findText(SUMMARY_PLACEHOLDER);
  if (!placeholderRange) {
    placeholderRange = body.appendParagraph(SUMMARY_PLACEHOLDER).editAsText();
  }

  const placeholderElement = placeholderRange.getElement().asParagraph();
  const insertionIndex = body.getChildIndex(placeholderElement);
  placeholderElement.removeFromParent();

  const heading = body.insertParagraph(insertionIndex, 'Sommario');
  heading.setHeading(DocumentApp.ParagraphHeading.HEADING2);

  const table = body.insertTable(insertionIndex + 1, [['Tipo', 'Quantita']]);
  summaryRows.forEach(row => {
    const tableRow = table.appendTableRow();
    row.forEach(value => tableRow.appendTableCell(value));
  });

  body.insertParagraph(insertionIndex + 2, '');
}

function capitalize(str) {
  return str.charAt(0).toUpperCase() + str.slice(1);
}


## KankaAPI.js


In [ ]:
const entityCache = {}; // cache in memoria per questa esecuzione
const MAX_RETRIES = 3;
const INITIAL_RETRY_DELAY = 1000; // 1 secondo

/**
 * Esegue una richiesta HTTP con meccanismo di retry e backoff esponenziale
 * @param {string} url - L'URL a cui effettuare la richiesta
 * @param {Object} options - Le opzioni della richiesta (metodo, headers, ecc.)
 * @param {number} [maxRetries=MAX_RETRIES] - Numero massimo di tentativi
 * @param {number} [retryDelay=INITIAL_RETRY_DELAY] - Ritardo iniziale tra i tentativi in ms
 * @returns {Object} L'oggetto risposta HTTP
 * @throws {Error} Se la richiesta fallisce dopo tutti i tentativi
 */
function fetchWithRetry(url, options, maxRetries = MAX_RETRIES, retryDelay = INITIAL_RETRY_DELAY) {
  let lastError;
  
  for (let attempt = 0; attempt < maxRetries; attempt++) {
    try {
      const response = UrlFetchApp.fetch(url, { ...options, muteHttpExceptions: true });
      const statusCode = response.getResponseCode();
      
      // Se la risposta è un successo (2xx) o un errore client (4xx) che non richiede retry
      if (statusCode < 400 || (statusCode >= 400 && statusCode < 500 && statusCode !== 408 && statusCode !== 429)) {
        return response;
      }
      
      // Se raggiungiamo qui, la richiesta ha avuto un errore che potrebbe essere temporaneo
      lastError = new Error(`HTTP ${statusCode}: ${response.getContentText()}`);
      lastError.statusCode = statusCode;
      
      // Se è un errore di rate limit, attendiamo il tempo specificato nell'header Retry-After
      if (statusCode === 429) {
        const retryAfter = response.getHeaders()['retry-after'] || 60; // default a 60 secondi
        Utilities.sleep(retryAfter * 1000);
        continue;
      }
    } catch (e) {
      lastError = e;
    }
    
    // Calcola il prossimo ritardo con backoff esponenziale
    const delay = retryDelay * Math.pow(2, attempt);
    Logger.log(`Tentativo ${attempt + 1} fallito. Nuovo tentativo tra ${delay}ms`);
    Utilities.sleep(delay);
  }
  
  throw lastError || new Error('Richiesta fallita senza un messaggio di errore specifico');
}

/**
 * Recupera tutte le entità di un determinato tipo dal server Kanka
 * @param {string} type - Il tipo di entità da recuperare (es. 'characters', 'locations', ecc.)
 * @returns {Array} Un array di entità
 * @throws {Error} Se si verifica un errore durante il recupero delle entità
 */
function fetchAllEntities(type) {
  const token = getApiToken();
  const campaignId = getCampaignId();
  const baseUrl = `https://api.kanka.io/1.0/campaigns/${campaignId}/${type}`;
  const headers = {
    'Authorization': `Bearer ${token}`,
    'Accept': 'application/json'
  };

  let results = [];
  let nextUrl = baseUrl;
  let page = 1;
  let totalItems = 0;

  try {
    while (nextUrl) {
      Logger.log(`Recupero pagina ${page} per ${type}...`);
      const response = fetchWithRetry(nextUrl, { 
        method: 'get', 
        headers 
      });
      
      const json = JSON.parse(response.getContentText());
      
      if (!Array.isArray(json.data)) {
        throw new Error(`Risposta API non valida per ${type}. Dati mancanti o non validi.`);
      }
      
      results.push(...json.data);
      totalItems = json.meta ? json.meta.total : results.length;
      nextUrl = json.links && json.links.next ? json.links.next : null;
      page++;
      
      // Breve pausa tra le richieste per evitare rate limiting
      Utilities.sleep(200);
    }

    Logger.log(`Recuperate ${results.length} di ${totalItems} ${type} in ${page - 1} pagine`);
    rememberEntityNames(type, results);
    
    return results;
  } catch (error) {
    Logger.log(`Errore durante il recupero delle entità di tipo ${type}: ${error.message}`);
    throw new Error(`Impossibile recuperare le entità di tipo ${type}: ${error.message}`);
  }
}

/**
 * Recupera tutti i personaggi dalla campagna Kanka
 * @returns {Array} Un array di personaggi
 * @throws {Error} Se si verifica un errore durante il recupero dei personaggi
 */
function fetchCharacters() {
  try {
    const characters = fetchAllEntities('characters');
    Logger.log(`Totale personaggi trovati: ${characters.length}`);
    return characters;
  } catch (error) {
    Logger.log(`Errore durante il recupero dei personaggi: ${error.message}`);
    throw new Error(`Impossibile recuperare i personaggi: ${error.message}`);
  }
}

/**
 * Recupera un'entità specifica dal server Kanka
 * @param {string} type - Il tipo di entità da recuperare (es. 'characters', 'locations', ecc.)
 * @param {number|string} id - L'ID dell'entità da recuperare
 * @returns {Object|null} L'entità richiesta o null se non trovata
 * @throws {Error} Se si verifica un errore durante il recupero dell'entità
 */
function fetchEntity(type, id) {
  const cacheKey = `${type}:${id}`;
  
  // Controlla se l'entità è già in cache
  if (Object.prototype.hasOwnProperty.call(entityCache, cacheKey)) {
    return entityCache[cacheKey];
  }

  const token = getApiToken();
  const campaignId = getCampaignId();
  const baseUrl = `https://api.kanka.io/1.0/campaigns/${campaignId}`;
  const headers = { 
    'Authorization': `Bearer ${token}`, 
    'Accept': 'application/json' 
  };

  // Prova prima l'endpoint specifico per il tipo, poi l'endpoint generico
  const urls = [
    `${baseUrl}/${type}/${id}`,
    `${baseUrl}/entities/${id}`
  ];

  for (const url of urls) {
    try {
      Logger.log(`Recupero entità da ${url}...`);
      const response = fetchWithRetry(url, { 
        method: 'get', 
        headers 
      });
      
      const json = JSON.parse(response.getContentText());
      
      if (json.data) {
        // Salva in cache e restituisci i dati
        entityCache[cacheKey] = json.data;
        return json.data;
      }
    } catch (error) {
      // Se riceviamo un 404, proviamo con l'URL successivo
      if (error.statusCode === 404) {
        continue;
      }
      
      // Per altri errori, logghiamo e rilanciamo
      Logger.log(`Errore durante il recupero dell'entità ${type}/${id}: ${error.message}`);
      throw new Error(`Impossibile recuperare l'entità ${type}/${id}: ${error.message}`);
    }
  }

  // Se arriviamo qui, l'entità non è stata trovata
  Logger.log(`Entità ${type}/${id} non trovata`);
  entityCache[cacheKey] = null;
  return null;
}


function fetchEntityName(type, id) {
  const entity = fetchEntity(type, id);
  return entity && entity.name ? entity.name : `[${type}:${id}]`;
}

/**
 * Recupera i dati completi di un'entità dal server Kanka
 * @param {number|string} entityId - L'ID dell'entità da recuperare
 * @returns {Object|null} I dati dell'entità o null se non trovata
 * @throws {Error} Se si verifica un errore durante il recupero dei dati
 */
function fetchEntityData(entityId) {
  const token = getApiToken();
  const headers = {
    'Authorization': `Bearer ${token}`,
    'Accept': 'application/json'
  };

  const url = `https://api.kanka.io/1.0/entities/${entityId}`;
  Logger.log(`Recupero dati entità ${entityId} da ${url}...`);
  
  try {
    const response = fetchWithRetry(url, {
      method: 'get',
      headers
    });
    
    const json = JSON.parse(response.getContentText());
    
    if (json.data) {
      const data = json.data;
      
      // Se l'entità ha un tipo e un nome, lo salviamo nella cache dei nomi
      if (data.id && data.name) {
        const entityType = data.type || 'entities';
        storeEntityName(entityType, data.id, data.name);
        Logger.log(`Trovata entità ${entityType}/${data.id}: ${data.name}`);
      }
      
      return data;
    }
    
    Logger.log(`Nessun dato trovato per l'entità ${entityId}`);
    return null;
  } catch (error) {
    Logger.log(`Errore durante il recupero dei dati per l'entità ${entityId}: ${error.message}`);
    
    // Se l'errore è un 404, l'entità non esiste
    if (error.statusCode === 404) {
      Logger.log(`Entità ${entityId} non trovata`);
      return null;
    }
    
    // Per altri errori, rilanciamo l'eccezione
    throw new Error(`Impossibile recuperare i dati per l'entità ${entityId}: ${error.message}`);
  }
}

## EntityCache.js


In [ ]:
const entityMaps = {};
const missingReferenceKeys = new Set();

const referenceTypeAliases = {
  character: 'characters',
  characters: 'characters',
  creature: 'creatures',
  creatures: 'creatures',
  location: 'locations',
  locations: 'locations',
  family: 'families',
  families: 'families',
  organisation: 'organisations',
  organisations: 'organisations',
  organization: 'organisations',
  organizations: 'organisations',
  race: 'races',
  races: 'races',
  tag: 'tags',
  tags: 'tags',
  item: 'items',
  items: 'items',
  event: 'events',
  events: 'events',
  journal: 'journals',
  journals: 'journals',
  quest: 'quests',
  quests: 'quests',
  ability: 'abilities',
  abilities: 'abilities',
  calendar: 'calendars',
  calendars: 'calendars',
  timeline: 'timelines',
  timelines: 'timelines',
  note: 'journals',
  notes: 'journals',
  entity: 'entities',
  entities: 'entities'
};

function normalizeReferenceType(type) {
  if (!type) return null;
  const key = String(type).toLowerCase();
  return referenceTypeAliases[key] || key;
}

function ensureEntityMap(type) {
  if (!type) return;
  const normalized = normalizeReferenceType(type);
  if (!normalized) return;
  if (!entityMaps[normalized]) {
    entityMaps[normalized] = new Map();
  }
}

function storeEntityName(type, id, name) {
  const normalized = normalizeReferenceType(type);
  if (!normalized || !id || !name) return;
  ensureEntityMap(normalized);
  entityMaps[normalized].set(String(id), String(name));
}

function rememberEntityNames(type, entities) {
  if (!Array.isArray(entities)) return;
  const normalized = normalizeReferenceType(type);
  if (!normalized) return;
  ensureEntityMap(normalized);
  entities.forEach(entity => {
    if (entity && entity.id && entity.name) {
      storeEntityName(normalized, entity.id, entity.name);
    }
  });
}

function preloadEntityMaps(types) {
  const input = Array.isArray(types) && types.length ? types : Object.keys(entityMaps);
  const normalizedTypes = Array.from(
    new Set(
      input
        .map(normalizeReferenceType)
        .filter(Boolean)
    )
  );

  const dataByType = {};
  normalizedTypes.forEach(type => {
    ensureEntityMap(type);
    const entities = fetchAllEntities(type);
    dataByType[type] = entities;
    rememberEntityNames(type, entities);
    const count = entityMaps[type] ? entityMaps[type].size : 0;
    Logger.log(`Precaricati ${count} elementi da ${type}`);
  });

  return dataByType;
}

function resolveEntityName(type, id) {
  if (!id) return '';

  const normalized = normalizeReferenceType(type);
  const key = String(id);
  const map = normalized ? entityMaps[normalized] : undefined;

  if (map && map.has(key)) {
    return map.get(key);
  }

  const lookupType = normalized || type;
  const name = fetchEntityName(lookupType, id);
  if (name && !/^\[.+:\d+\]$/.test(name)) {
    return name;
  }

  return `[${type}:${id}]`;
}


## EntityRenderer.js


In [ ]:
function renderEntityByType(type, entity, doc) {
  if (type === 'calendars') {
    const full = fetchEntity(type, entity.id);
    if (full?.months && full?.weekdays) {
      formatCalendarData(doc, full);
      return;
    }
  }

  if (type === 'timelines') {
    const full = fetchEntity(type, entity.id);
    if (full?.eras?.length > 0) {
      formatTimelineData(doc, full);
      return;
    }
  }

  if (type === 'characters') {
    formatCharacterData(doc, entity); // da charactersParser.gs
    return;
  }

  // fallback generico - più ricco
  const full = fetchEntity(type, entity.id) || entity;
  const name = full.name || '(senza nome)';
  const raw = full.entry || '';
  const parsed = resolveReferences(raw);
  const clean = stripHtml(parsed);

  const body = doc.getBody();
  body.appendParagraph(name).setHeading(DocumentApp.ParagraphHeading.HEADING2);

  // immagine
  addEntityImagePlaceholder(body, type, full);

  // metadati (id, created/updated, owner)
  addMetadata(body, full);

  // Campi principali comuni
  const kv = [];
  if (full.title) kv.push({ key: 'Titolo', value: full.title });
  if (full.type) kv.push({ key: 'Tipo entità', value: full.type });
  if (full.age) kv.push({ key: 'Età', value: full.age });
  if (full.sex) kv.push({ key: 'Sesso', value: full.sex });
  if (full.pronouns) kv.push({ key: 'Pronomi', value: full.pronouns });

  // Riferimenti risolvibili
  if (full.location_id || full.location) {
    const locationName = full.location_name || (full.location_id ? resolveEntityName('locations', full.location_id) : full.location);
    kv.push({ key: 'Posizione', value: locationName });
  }
  if (full.family_id || full.family) {
    const familyName = full.family_name || (full.family_id ? resolveEntityName('families', full.family_id) : full.family);
    kv.push({ key: 'Famiglia', value: familyName });
  }
  if (full.race_id || full.race) {
    const raceName = full.race_name || (full.race_id ? resolveEntityName('races', full.race_id) : full.race);
    kv.push({ key: 'Razza', value: raceName });
  }

  if (kv.length > 0) {
    addSectionTitle(body, "🔎 Informazioni principali", 3);
    addKeyValueList(body, kv);
  }

  // Tags
  if (Array.isArray(full.tags) && full.tags.length > 0) {
    addSectionTitle(body, "🏷️ Tags", 3);
    full.tags.forEach(t => {
      const name = (t && (t.name || t)) ? (t.name || t) : String(t);
      body.appendParagraph(`• ${name}`);
    });
  } else if (full.tag_list) {
    addSectionTitle(body, "🏷️ Tags", 3);
    body.appendParagraph(full.tag_list);
  }

  // entry pulito
  if (clean) {
    addSectionTitle(body, "📝 Descrizione", 3);
    body.appendParagraph(clean);
  }

  // Attributi personalizzati se presenti come array
  if (Array.isArray(full.attributes) && full.attributes.length > 0) {
    addSectionTitle(body, "🧩 Attributi personalizzati", 3);
    full.attributes.forEach(a => {
      body.appendParagraph(`• ${a.name}: ${a.value}`);
    });
  }

  // Lista generale di proprietà utili (fallback)
  const extras = [];
  ['size', 'moral', 'status', 'age', 'height', 'weight', 'history', 'appearance'].forEach(k => {
    if (full[k]) extras.push({ key: capitalize(k), value: stripHtml(resolveReferences(String(full[k] || ''))) });
  });
  if (extras.length > 0) {
    addSectionTitle(body, "🔧 Altri dettagli", 3);
    addKeyValueList(body, extras);
  }

  body.appendParagraph('');
}


## Formatter.js


In [ ]:
function resolveReferences(text) {
  if (!text) return '';
  const regex = /\[([a-z_]+):(\d+)\]/gi;

  return text.replace(regex, (match, type, id) => {
    const resolved = resolveEntityName(type, id);
    if (!resolved || resolved === `[${type}:${id}]`) {
      return match;
    }
    return resolved;
  });
}

function stripHtml(html) {
  if (!html) return '';
  return html.replace(/<[^>]*>/g, '').trim();
}

function addSectionTitle(body, text, level = 2) {
  const headingMap = {
    1: DocumentApp.ParagraphHeading.HEADING1,
    2: DocumentApp.ParagraphHeading.HEADING2,
    3: DocumentApp.ParagraphHeading.HEADING3
  };
  const paragraph = body.appendParagraph(text);
  paragraph.setHeading(headingMap[level] || DocumentApp.ParagraphHeading.NORMAL);
  return paragraph;
}

const IMAGE_PLACEHOLDER_REGEX = /\{\{IMAGE:([a-z_]+):(\d+)\}\}/i;

function addEntityImagePlaceholder(body, type, entity) {
  if (!entity?.image_full) {
    return;
  }

  const paragraph = body.appendParagraph(createImagePlaceholder(type, entity.id));
  paragraph.setForegroundColor('#888888');
  paragraph.setItalic(true);
}

function createImagePlaceholder(type, id) {
  return `{{IMAGE:${type}:${id}}}`;
}

function parseImagePlaceholder(text) {
  if (!text) return null;
  const match = IMAGE_PLACEHOLDER_REGEX.exec(text.trim());
  if (!match) return null;

  return {
    type: match[1],
    id: match[2]
  };
}

function addKeyValueList(body, entries) {
  // entries: array di { key, value } oppure oggetto semplice { key: value }
  if (!entries) return;
  if (!Array.isArray(entries)) {
    entries = Object.keys(entries).map(k => ({ key: k, value: entries[k] }));
  }
  entries.forEach(entry => {
    const value = (entry.value === null || entry.value === undefined || entry.value === '') ? '—' : String(entry.value);
    body.appendParagraph(`• ${entry.key}: ${value}`);
  });
}

// Aggiunge una sezione metadati comune per qualsiasi entità
function addMetadata(body, entity) {
  if (!entity) return;
  const meta = [];
  if (entity.id) meta.push({ key: "ID", value: entity.id });
  if (entity.type) meta.push({ key: "Tipo", value: entity.type });
  if (entity.slug) meta.push({ key: "Slug", value: entity.slug });
  if (entity.created_at) meta.push({ key: "Creato", value: formatDateTime(entity.created_at) });
  if (entity.updated_at) meta.push({ key: "Aggiornato", value: formatDateTime(entity.updated_at) });
  if (entity.created_by_name) meta.push({ key: "Creato da", value: entity.created_by_name });
  if (entity.owner_name) meta.push({ key: "Proprietario", value: entity.owner_name });

  if (meta.length > 0) {
    addSectionTitle(body, "ℹ️ Metadati", 3);
    addKeyValueList(body, meta);
  }
}

function formatDateTime(dt) {
  if (!dt) return '';
  // Kanka spesso usa ISO; proviamo il parsing semplice
  try {
    const d = new Date(dt);
    if (isNaN(d.getTime())) return dt;
    return d.toLocaleString();
  } catch (e) {
    return dt;
  }
}

function addTable(body, headers, rows) {
  const table = body.appendTable();

  // Add header cells
  const headerRow = table.appendTableRow();
  headers.forEach(h => headerRow.appendTableCell(String(h)));

  // Add data rows
  rows.forEach(row => {
    const tr = table.appendTableRow();
    row.forEach(cell => tr.appendTableCell(String(cell)));
  });
}

function formatCalendarData(doc, entity) {
  const body = doc.getBody();
  addSectionTitle(body, entity.name || "Calendario");
  addEntityImagePlaceholder(body, 'calendars', entity);

  body.appendParagraph(`📅 Data di riferimento: ${formatCalendarDate(entity.date)}`);

  // Mesi
  addSectionTitle(body, "📘 Mesi dell’anno", 3);
  addTable(body, ["Nome", "Giorni", "Alias", "Tipo"],
    entity.months.map(m => [m.name, `${m.length}`, m.alias || "-", m.type]));

  // Giorni settimana
  addSectionTitle(body, "📆 Giorni della settimana", 3);
  entity.weekdays.forEach(day => body.appendParagraph(`• ${day}`));

  // Stagioni
  addSectionTitle(body, "🌸 Stagioni", 3);
  entity.seasons.forEach(s =>
    body.appendParagraph(`• ${s.name} – giorno ${s.day} del mese ${s.month}`));

  // Lune
  addSectionTitle(body, "🌙 Lune", 3);
  addTable(body, ["Nome", "Luna piena", "Offset", "Colore"],
    entity.moons.map(m => [m.name, `${m.fullmoon}`, `${m.offset}`, m.colour]));

  // Eventi
  if (entity.entity_events?.length) {
    addSectionTitle(body, "🪧 Eventi ricorrenti", 3);
    entity.entity_events.forEach(ev => {
      body.appendParagraph(`• ${ev.comment} – ${formatCalendarDate(ev.date)}${ev.is_recurring ? " (ricorrente)" : ""}`);
    });
  }
}

function formatCalendarDate(dateStr) {
  const parts = dateStr.split("-");
  if (parts.length !== 3) return dateStr;
  const [year, month, day] = parts;
  const monthNames = [
    "Lairesa", "Kinera", "Eldar", "Nivara", "Flora", "Solara",
    "Ignara", "Altara", "Venta", "Folia", "Mora", "Nocta"
  ];
  return `${parseInt(day)} ${monthNames[parseInt(month) - 1]} ${year}`;
}

function formatTimelineData(doc, entity) {
  const body = doc.getBody();
  addSectionTitle(body, entity.name || "📜 Timeline", 2);
  addEntityImagePlaceholder(body, 'timelines', entity);

  if (!entity.eras || entity.eras.length === 0) {
    body.appendParagraph("⚠️ Nessuna era presente in questa timeline.");
    return;
  }

  entity.eras.forEach(era => {
    // Titolo era
    const eraLabel = `🕰️ ${era.name}` + (era.abbreviation ? ` (${era.abbreviation})` : '');
    addSectionTitle(body, eraLabel, 3);

    // Periodo dell'era (se noto)
    if (era.start_year || era.end_year) {
      const start = era.start_year !== null ? era.start_year : "???";
      const end = era.end_year !== null ? era.end_year : "???";
      body.appendParagraph(`📅 Periodo: dal ${start} al ${end}`);
    }

    // Descrizione dell'era
    if (era.entry) {
      body.appendParagraph(stripHtml(resolveReferences(era.entry)));
    }

    // Eventi della timeline (elements)
    if (Array.isArray(era.elements) && era.elements.length > 0) {
      era.elements
        .sort((a, b) => a.position - b.position)
        .forEach(element => {
          const date = element.date ? `(${element.date})` : '';
          const title = element.name?.trim() || '';
          const entry = stripHtml(resolveReferences(element.entry || ''));

          const bulletTitle = title ? `• ${title} ${date}` : `• ${date}`;

          addSectionTitle(body, bulletTitle.trim(), 4);
          if (entry) body.appendParagraph(entry);
        });
    } else {
      body.appendParagraph("• Nessun evento registrato per quest'era.");
    }

    body.appendParagraph(""); // spazio extra
  });
}


## ImageSync.js


In [ ]:
const IMAGE_PLACEHOLDER_MESSAGE = '🖼 Sincronizzazione immagini completata';

function syncEntityImages() {
  checkRequiredConfig();

  const docId = getMasterDocumentId();
  if (!docId) {
    Logger.log('❌ Documento principale non configurato.');
    return;
  }

  const doc = DocumentApp.openById(docId);
  const body = doc.getBody();
  const paragraphs = body.getParagraphs();
  let updated = 0;
  let skipped = 0;

  paragraphs.forEach(paragraph => {
    const originalText = paragraph.getText();
    const placeholder = parseImagePlaceholder(originalText);
    if (!placeholder) {
      return;
    }

    const trimmed = (originalText || '').trim();
    const expectedPlaceholder = createImagePlaceholder(placeholder.type, placeholder.id);
    if (trimmed !== expectedPlaceholder) {
      Logger.log(`⚠️ Segnaposto immagine ignorato per evitare la perdita di testo: "${trimmed}"`);
      return;
    }

    const inserted = insertImageFromPlaceholder(paragraph, placeholder.type, placeholder.id);
    if (inserted) {
      updated++;
    } else {
      skipped++;
      ensurePlaceholderFormatting(paragraph, expectedPlaceholder);
    }
  });

  Logger.log(`${IMAGE_PLACEHOLDER_MESSAGE}: ${updated} inserite, ${skipped} senza immagine`);
}

function insertImageFromPlaceholder(paragraph, type, id) {
  const entity = fetchEntity(type, id);
  const name = entity?.name || '(senza nome)';
  const entityLabel = entity ? `${name} (${type}:${id})` : `${type}:${id}`;
  const preparedBlob = getEntityImageBlob(entity, type, id);

  if (!preparedBlob) {
    Logger.log(`ℹ️ Nessuna immagine disponibile per ${entityLabel}`);
    return false;
  }

  try {
    paragraph.clear();
    paragraph.appendInlineImage(preparedBlob);
    formatInsertedImage(paragraph);
    return true;
  } catch (e) {
    Logger.log(`❌ Errore nell'inserimento dell'immagine per ${entityLabel}: ${e}`);
    return false;
  }
}

function formatInsertedImage(paragraph) {
  try {
    paragraph.setAlignment(DocumentApp.HorizontalAlignment.CENTER);
    paragraph.setSpacingBefore(6);
    paragraph.setSpacingAfter(6);
    paragraph.setLineSpacing(1);
  } catch (formatError) {
    Logger.log(`⚠️ Impossibile applicare la formattazione del paragrafo per l'immagine: ${formatError}`);
  }
}

function ensurePlaceholderFormatting(paragraph, placeholderText) {
  const currentText = (paragraph.getText() || '').trim();
  const textElement = paragraph.editAsText();
  if (!textElement) {
    return;
  }

  if (currentText !== placeholderText) {
    textElement.setText(placeholderText);
  }

  const length = textElement.getText().length;
  if (length === 0) {
    return;
  }

  textElement.setItalic(0, length - 1, true);
  textElement.setForegroundColor(0, length - 1, '#888888');
}

function getEntityImageBlob(entity, type, id) {
  if (!entity) {
    return null;
  }

  const name = entity.name || '(senza nome)';
  const entityLabel = `${name} (${type}:${id})`;
  const filenameHint = `${type}-${id}`;
  const variants = [];

  if (entity.image_full) {
    variants.push({ url: entity.image_full, label: 'immagine completa', allowTiny: true });
  }

  if (entity.image_thumb) {
    variants.push({ url: entity.image_thumb, label: 'miniatura', allowTiny: false });
  }

  if (variants.length === 0) {
    return null;
  }

  for (let i = 0; i < variants.length; i++) {
    const variant = variants[i];
    const blob = downloadImageBlob(variant.url, entityLabel, variant.label);
    if (!blob) {
      continue;
    }

    const prepared = prepareEntityImageBlob(blob, filenameHint, entityLabel, variant.allowTiny);
    if (prepared) {
      if (!variant.allowTiny) {
        Logger.log(`ℹ️ Utilizzo la miniatura Kanka per ${entityLabel}.`);
      }
      return prepared;
    }
  }

  return null;
}

function downloadImageBlob(url, entityLabel, description) {
  if (!url) {
    return null;
  }

  try {
    const response = UrlFetchApp.fetch(url, { muteHttpExceptions: true });
    const code = response.getResponseCode();
    if (code >= 400) {
      Logger.log(`⚠️ Impossibile scaricare ${description} per ${entityLabel} (HTTP ${code}): ${url}`);
      return null;
    }

    return response.getBlob();
  } catch (error) {
    Logger.log(`❌ Errore durante lo scaricamento di ${description} per ${entityLabel}: ${error}`);
    return null;
  }
}

function prepareEntityImageBlob(blob, filenameHint, entityLabel, allowTinyResize) {
  if (!blob) {
    return null;
  }

  let workingBlob = blob;
  if (allowTinyResize) {
    workingBlob = attemptTinyResize(workingBlob, entityLabel);
  }

  const preparedBlob = prepareImageBlob(workingBlob, filenameHint);
  if (!preparedBlob) {
    Logger.log(`⚠️ Dati immagine non utilizzabili per ${entityLabel}.`);
  }
  return preparedBlob;
}

function attemptTinyResize(blob, entityLabel) {
  if (!blob) {
    return blob;
  }

  if (typeof getTinyPngApiKey !== 'function') {
    return blob;
  }

  const apiKey = getTinyPngApiKey();
  if (!apiKey) {
    return blob;
  }

  const credentials = apiKey.startsWith('api:') ? apiKey : `api:${apiKey}`;
  const authHeader = 'Basic ' + Utilities.base64Encode(credentials);

  try {
    const shrinkResp = UrlFetchApp.fetch('https://api.tinify.com/shrink', {
      method: 'post',
      headers: { Authorization: authHeader },
      payload: blob,
      muteHttpExceptions: true,
    });
    const shrinkCode = shrinkResp.getResponseCode();
    if (shrinkCode >= 400) {
      Logger.log(`⚠️ Compressione TinyPNG fallita per ${entityLabel} (HTTP ${shrinkCode}): ${shrinkResp.getContentText()}`);
      return blob;
    }

    let tinyUrl = extractTinyOutputUrl(shrinkResp);
    if (!tinyUrl) {
      Logger.log(`⚠️ Risposta TinyPNG senza URL di output per ${entityLabel}.`);
      return blob;
    }

    const maxDimension = typeof getTinyPngMaxDimension === 'function' ? getTinyPngMaxDimension() : null;
    if (maxDimension && maxDimension > 0) {
      const resizeResp = UrlFetchApp.fetch(tinyUrl, {
        method: 'post',
        headers: {
          Authorization: authHeader,
          'Content-Type': 'application/json',
        },
        payload: JSON.stringify({ resize: { method: 'fit', width: maxDimension, height: maxDimension } }),
        muteHttpExceptions: true,
      });
      const resizeCode = resizeResp.getResponseCode();
      if (resizeCode >= 400) {
        Logger.log(`⚠️ Ridimensionamento TinyPNG fallito per ${entityLabel} (HTTP ${resizeCode}): ${resizeResp.getContentText()}`);
      } else {
        const resizedUrl = extractTinyOutputUrl(resizeResp);
        if (resizedUrl) {
          tinyUrl = resizedUrl;
        } else if (isTinyBinaryResponse(resizeResp)) {
          return resizeResp.getBlob();
        }
      }
    }

    const finalResp = UrlFetchApp.fetch(tinyUrl, {
      headers: { Authorization: authHeader },
      muteHttpExceptions: true,
    });
    const finalCode = finalResp.getResponseCode();
    if (finalCode >= 400) {
      Logger.log(`⚠️ Download TinyPNG fallito per ${entityLabel} (HTTP ${finalCode}): ${tinyUrl}`);
      return blob;
    }

    return finalResp.getBlob();
  } catch (error) {
    Logger.log(`⚠️ Impossibile ridimensionare l'immagine con TinyPNG per ${entityLabel}: ${error}`);
    return blob;
  }
}

function extractTinyOutputUrl(response) {
  if (!response) {
    return null;
  }

  const headers = response.getHeaders ? response.getHeaders() : {};
  if (headers) {
    const directLocation = headers.Location || headers.location;
    if (directLocation) {
      return directLocation;
    }
  }

  const text = response.getContentText();
  if (text) {
    try {
      const payload = JSON.parse(text);
      if (payload && payload.output && payload.output.url) {
        return payload.output.url;
      }
    } catch (jsonError) {
      // Ignoro la risposta non JSON
    }
  }

  return null;
}

function isTinyBinaryResponse(response) {
  if (!response || !response.getHeaders) {
    return false;
  }

  const headers = response.getHeaders();
  const contentType = (headers['Content-Type'] || headers['content-type'] || '').toLowerCase();
  return contentType.startsWith('image/');
}

function prepareImageBlob(blob, filenameHint) {
  if (!blob) {
    return null;
  }

  const bytes = blob.getBytes();
  if (!bytes || bytes.length === 0) {
    return null;
  }

  let contentType = (blob.getContentType() || '').toLowerCase();
  if (!contentType.startsWith('image/')) {
    return null;
  }

  const supportedTypes = [MimeType.PNG, MimeType.JPEG, MimeType.GIF, MimeType.BMP];
  let workingBlob = blob;

  if (!supportedTypes.includes(contentType)) {
    try {
      workingBlob = blob.getAs(MimeType.PNG);
      contentType = MimeType.PNG;
    } catch (conversionError) {
      Logger.log(`⚠️ Impossibile convertire l'immagine (${contentType}) in PNG: ${conversionError}`);
      return null;
    }
  }

  if (filenameHint) {
    const extension = contentType.split('/')[1] || 'png';
    workingBlob.setName(`${filenameHint}.${extension}`);
  }

  return workingBlob;
}


## CharactersParser.js


In [ ]:
function formatCharacterData(doc, entity) {
  const body = doc.getBody();
  body.clear();
  
  // Add character name as document title with death indicator if applicable
  const title = `${entity.name || '(senza nome)'}${entity.is_dead ? ' (Deceduto/a)' : ''}`;
  body.appendParagraph(title).setHeading(DocumentApp.ParagraphHeading.TITLE);
  
  // Add character image if available
  if (entity.image_full || entity.image_thumb) {
    addEntityImagePlaceholder(body, 'characters', entity);
  }
  
  // Helper function to add a section with title and content
  const addSection = (tabBody, title, content, level = 3) => {
    if (content && content.trim() !== '') {
      addSectionTitle(tabBody, title, level);
      tabBody.appendParagraph(content);
      return true;
    }
    return false;
  };
  
  // --- PANORAMICA TAB ---
  const overviewBody = body;
  
  // Add metadata (creator, last update, etc.)
  addMetadata(overviewBody, entity);
  
  // Main information section
  const mainInfo = [];
  if (entity.title) mainInfo.push({ key: 'Titolo', value: entity.title });
  if (entity.type) mainInfo.push({ key: 'Tipo', value: entity.type });
  if (entity.age) mainInfo.push({ key: 'Età', value: entity.age });
  if (entity.sex) mainInfo.push({ key: 'Sesso', value: entity.sex });
  if (entity.pronouns) mainInfo.push({ key: 'Pronomi', value: entity.pronouns });
  
  // Handle races (both public and private)
  const races = [];
  if (entity.race_name) races.push(entity.race_name);
  if (entity.race_id && !entity.race_name) {
    const raceName = resolveEntityName('races', entity.race_id);
    if (raceName) races.push(raceName);
  }
  if (entity.races && entity.races.length > 0) {
    entity.races.forEach(raceId => {
      const raceName = resolveEntityName('races', raceId);
      if (raceName) races.push(raceName);
    });
  }
  if (entity.private_races && entity.private_races.length > 0) {
    entity.private_races.forEach(raceId => {
      const raceName = resolveEntityName('races', raceId);
      if (raceName) races.push(`${raceName} (privato)`);
    });
  }
  if (races.length > 0) {
    mainInfo.push({ key: 'Razza/e', value: races.join(', ') });
  }

  // Family information
  const families = [];
  if (entity.family_name) families.push(entity.family_name);
  if (entity.family_id && !entity.family_name) {
    const familyName = resolveEntityName('families', entity.family_id);
    if (familyName) families.push(familyName);
  }
  if (entity.families && entity.families.length > 0) {
    entity.families.forEach(familyId => {
      const familyName = resolveEntityName('families', familyId);
      if (familyName) families.push(familyName);
    });
  }
  if (families.length > 0) {
    mainInfo.push({ 
      key: families.length > 1 ? 'Famiglie' : 'Famiglia', 
      value: families.join(', ') 
    });
  }

  // Location information
  if (entity.location_name) {
    mainInfo.push({ key: 'Posizione', value: entity.location_name });
  } else if (entity.location_id) {
    const locationName = resolveEntityName('locations', entity.location_id);
    if (locationName) {
      mainInfo.push({ key: 'Posizione', value: locationName });
    }
  }

  // Add main info to overview
  if (mainInfo.length > 0) {
    addSectionTitle(overviewBody, "🔎 Informazioni principali", 3);
    addKeyValueList(overviewBody, mainInfo);
  }
  
  // Add biography with proper HTML handling
  const rawBio = entity.entry || '';
  const parsedBio = resolveReferences(rawBio);
  const cleanBio = stripHtml(parsedBio);
  addSection(overviewBody, "📝 Biografia", cleanBio);
  
  // Add appearance if available
  if (entity.appearance) {
    const cleanAppearance = stripHtml(resolveReferences(entity.appearance));
    addSection(overviewBody, "👤 Aspetto fisico", cleanAppearance);
  }
  
  // Add history if available
  if (entity.history) {
    const cleanHistory = stripHtml(resolveReferences(entity.history));
    addSection(overviewBody, "📜 Storia", cleanHistory);
  }
  
  // --- ATTRIBUTI TAB ---
  const attributesBody = body;
  
  // Add custom attributes
  if (entity.attributes && entity.attributes.length > 0) {
    addSectionTitle(attributesBody, "🧬 Attributi personalizzati", 2);
    const groupedAttributes = {};
    
    // Group attributes by section
    entity.attributes.forEach(attr => {
      const section = attr.section || 'Generali';
      if (!groupedAttributes[section]) {
        groupedAttributes[section] = [];
      }
      groupedAttributes[section].push(attr);
    });
    
    // Add each section
    Object.entries(groupedAttributes).forEach(([section, attrs]) => {
      addSectionTitle(attributesBody, `📋 ${section}`, 3);
      attrs.forEach(attr => {
        attributesBody.appendParagraph(`• ${attr.name}: ${attr.value}`);
      });
    });
  } else {
    attributesBody.appendParagraph("Nessun attributo personalizzato trovato.")
      .setItalic(true);
  }
  
  // --- PERSONALITÀ TAB ---
  const personalityBody = body;
  
  // Process traits if available
  if (Array.isArray(entity.traits) && entity.traits.length > 0) {
    const traitsBySection = {};
    
    // Group traits by section
    entity.traits.forEach(trait => {
      const section = trait.section || 'Generale';
      if (!traitsBySection[section]) {
        traitsBySection[section] = [];
      }
      traitsBySection[section].push(trait);
    });
    
    // Add each trait section
    Object.entries(traitsBySection).forEach(([section, traits]) => {
      addSectionTitle(personalityBody, `✨ ${section}`, 3);
      traits.forEach(trait => {
        const entry = trait.entry_parsed ? stripHtml(trait.entry_parsed) : trait.entry;
        personalityBody.appendParagraph(`• ${trait.name}: ${entry || '(nessun dettaglio)'}`);
      });
    });
  } else if (Array.isArray(entity.tags) && entity.tags.length > 0) {
    addSectionTitle(personalityBody, "🏷️ Tags", 2);
    entity.tags.forEach(tag => {
      personalityBody.appendParagraph(`• ${tag.name || tag}`);
    });
  } else {
    personalityBody.appendParagraph("Nessun tratto o tag trovato.")
      .setItalic(true);
  }
  
  // --- RELAZIONI TAB ---
  const relationshipsBody = body;
  
  if (entity.relations && entity.relations.length > 0) {
    // Group relations by type
    const relationsByType = {};
    entity.relations.forEach(rel => {
      const type = rel.relation || rel.type || 'Generica';
      if (!relationsByType[type]) {
        relationsByType[type] = [];
      }
      relationsByType[type].push(rel);
    });
    
    // Add each relation type
    Object.entries(relationsByType).forEach(([type, relations]) => {
      addSectionTitle(relationshipsBody, `🤝 ${type}`, 3);
      relations.forEach(rel => {
        const target = rel.target_name || 
                     (rel.target_id ? resolveEntityName(rel.target_entity_type || 'characters', rel.target_id) : '[sconosciuto]');
        const details = rel.attitude ? ` (${rel.attitude})` : '';
        relationshipsBody.appendParagraph(`• ${rel.relation || 'Relazione'} con ${target}${details}`);
        
        if (rel.attributions) {
          relationshipsBody.appendParagraph(`  - Attribuzioni: ${rel.attributions}`)
            .setIndentStart(20);
        }
      });
    });
  } else {
    relationshipsBody.appendParagraph("Nessuna relazione registrata.")
      .setItalic(true);
  }
  
  // --- INVENTARIO TAB ---
  const inventoryBody = body;
  
  if (entity.inventory && entity.inventory.length > 0) {
    // Group inventory by location if available
    const inventoryByLocation = {};
    entity.inventory.forEach(item => {
      const location = item.location_id ? 
        resolveEntityName('locations', item.location_id) : 'Generale';
      if (!inventoryByLocation[location]) {
        inventoryByLocation[location] = [];
      }
      inventoryByLocation[location].push(item);
    });
    
    // Add each location's items
    Object.entries(inventoryByLocation).forEach(([location, items]) => {
      addSectionTitle(inventoryBody, `📍 ${location}`, 3);
      items.forEach(item => {
        const name = item.name || item.entity?.name || item.entity_name || 'Oggetto';
        const amount = item.amount ? ` (${item.amount})` : '';
        const position = item.position ? ` [${item.position}]` : '';
        const notes = item.notes ? ` — ${stripHtml(resolveReferences(item.notes))}` : '';
        inventoryBody.appendParagraph(`• ${name}${amount}${position}${notes}`);
      });
    });
  } else {
    inventoryBody.appendParagraph("L'inventario è vuoto.")
      .setItalic(true);
  }
  
  // --- NOTE & ALTRO TAB ---
  const notesBody = body;
  
  // Add private notes if available and visible
  if (entity.private_notes && entity.is_private_visible !== false) {
    const cleanPrivateNotes = stripHtml(resolveReferences(entity.private_notes));
    addSection(notesBody, "🔒 Note private", cleanPrivateNotes, 2);
  }
  
  // Add tags if any
  if (Array.isArray(entity.tags) && entity.tags.length > 0) {
    addSectionTitle(notesBody, "🏷️ Tags", 2);
    notesBody.appendParagraph(
      entity.tags.map(t => t.name || t).join(' • ')
    );
  }
  
  // Add template information if this is a template
  if (entity.is_template) {
    addSectionTitle(notesBody, "📋 Modello", 2);
    notesBody.appendParagraph("Questo personaggio è un modello utilizzabile per creare altri personaggi.");
  }
  
  // Add visibility information
  const visibility = [];
  if (entity.is_private) visibility.push("Privato");
  if (entity.is_personality_visible === false) visibility.push("Personalità nascosta");
  if (visibility.length > 0) {
    addSectionTitle(notesBody, "👁️ Visibilità", 2);
    notesBody.appendParagraph(visibility.join(' • '));
  }
  
  // Add a small footer with last update info on the first page
  const lastUpdate = entity.updated_at || entity.created_at;
  if (lastUpdate) {
    const formattedDate = Utilities.formatDate(
      new Date(lastUpdate), 
      Session.getScriptTimeZone(), 
      "dd/MM/yyyy 'alle' HH:mm"
    );
    
    const footer = body.appendParagraph(`\n\nUltimo aggiornamento: ${formattedDate}`);
    footer.setFontSize(8).setForegroundColor('#666666');
    
    // Add creator/updater info if available
    if (entity.created_by_name || entity.updated_by_name) {
      const creator = entity.created_by_name ? `Creato da ${entity.created_by_name}` : '';
      const updater = entity.updated_by_name && entity.updated_by_name !== entity.created_by_name ? 
        ` • Ultima modifica di ${entity.updated_by_name}` : '';
      
      if (creator || updater) {
        const credits = body.appendParagraph(`${creator}${updater}`);
        credits.setFontSize(8).setForegroundColor('#666666');
      }
    }
  }
}

## Config.js


In [ ]:
const MASTER_DOCUMENT_KEY = 'GOOGLE_DOC_ID';
const DEFAULT_IMAGE_DISPLAY_WIDTH_INCHES = 3;
const POINTS_PER_INCH = 72;
const TINYPNG_API_KEY_KEY = 'TINYPNG_API_KEY';
const TINYPNG_MAX_DIMENSION_KEY = 'TINYPNG_MAX_DIMENSION';
const DEFAULT_TINYPNG_MAX_DIMENSION = 800;
const MAX_TINYPNG_DIMENSION = 4096;

function getApiToken() {
  return PropertiesService.getScriptProperties().getProperty('KANKA_API_TOKEN');
}

function getCampaignId() {
  return PropertiesService.getScriptProperties().getProperty('CAMPAIGN_ID');
}

function getMasterDocumentId() {
  return PropertiesService.getScriptProperties().getProperty(MASTER_DOCUMENT_KEY);
}

function getPreferredImageThumbSize() {
  const configured = PropertiesService.getScriptProperties().getProperty(IMAGE_THUMB_SIZE_KEY);
  if (!configured) {
    return DEFAULT_IMAGE_THUMB_SIZE;
  }

  const normalized = configured.trim();
  const sizePattern = /^\d+x\d+$/;
  if (!sizePattern.test(normalized)) {
    Logger.log(`⚠️ Valore non valido per ${IMAGE_THUMB_SIZE_KEY}: "${configured}". Uso il default ${DEFAULT_IMAGE_THUMB_SIZE}.`);
    return DEFAULT_IMAGE_THUMB_SIZE;
  }

  return normalized;
}

function checkRequiredConfig() {
  const token = getApiToken();
  const campaignId = getCampaignId();
  const docId = getMasterDocumentId(); // <-- usa questa ora

  const missing = [];

  if (!token) missing.push('KANKA_API_TOKEN');
  if (!campaignId) missing.push('CAMPAIGN_ID');
  if (!docId) missing.push('GOOGLE_DOC_ID');

  Logger.log('Configurazione:');
  Logger.log(' - Token API: %s', token ? 'OK' : 'MANCANTE');
  Logger.log(' - ID Campagna: %s', campaignId ? 'OK' : 'MANCANTE');
  Logger.log(' - ID Google Doc: %s', docId ? 'OK' : 'MANCANTE');

  if (missing.length > 0) {
    throw new Error('Variabili mancanti: ' + missing.join(', '));
  }
}

function getDocumentIdForType(type) {
  const key = ENTITY_DOCUMENT_KEYS[type];
  if (!key) {
    Logger.log(`❌ Tipo non supportato per document ID: ${type}`);
    return null;
  }
  return PropertiesService.getScriptProperties().getProperty(key);
}

function getTinyPngApiKey() {
  const key = PropertiesService.getScriptProperties().getProperty(TINYPNG_API_KEY_KEY);
  if (!key) {
    return null;
  }
  const normalized = key.trim();
  return normalized ? normalized : null;
}

function getTinyPngMaxDimension() {
  const configured = PropertiesService.getScriptProperties().getProperty(TINYPNG_MAX_DIMENSION_KEY);
  if (!configured) {
    return DEFAULT_TINYPNG_MAX_DIMENSION;
  }

  const parsed = parseInt(configured, 10);
  if (isNaN(parsed) || parsed <= 0) {
    Logger.log(`⚠️ Valore non valido per ${TINYPNG_MAX_DIMENSION_KEY}: "${configured}". Uso il default ${DEFAULT_TINYPNG_MAX_DIMENSION}.`);
    return DEFAULT_TINYPNG_MAX_DIMENSION;
  }

  if (parsed > MAX_TINYPNG_DIMENSION) {
    Logger.log(`ℹ️ Limito ${TINYPNG_MAX_DIMENSION_KEY} a ${MAX_TINYPNG_DIMENSION} px.`);
  }

  return Math.min(parsed, MAX_TINYPNG_DIMENSION);
}
